In [ ]:
# ============================================================
# FASE 3 - LABEL ENGINEERING
# Cell 1: Setup + Temporal Backward-Labeling + Validasi
# ============================================================

import sys
import numpy as np
import pandas as pd
from pathlib import Path

# ----------------------------------------------------------
# [1] SYS.PATH SETUP
# Notebook berada di: notebooks/fase_3_label_engineering/
# ML_ROOT = dua level ke atas
# ----------------------------------------------------------
NOTEBOOK_DIR = Path().resolve()
ML_ROOT      = NOTEBOOK_DIR.parent.parent
SRC_PATH     = ML_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

# ----------------------------------------------------------
# [2] IMPORT CONFIG - single source of truth
# ----------------------------------------------------------
from config import (
    SENSOR_FILE,
    MAINTENANCE_FILE,
    GLOBAL_SEED,
    W_CRITICAL_HRS,
    W_WARNING_HRS,
    LABEL_MAP,
)

np.random.seed(GLOBAL_SEED)

# ----------------------------------------------------------
# [3] LOAD DATA
# ----------------------------------------------------------
df_sensor = pd.read_csv(
    SENSOR_FILE,
    parse_dates=["timestamp"],
    dtype={"machine_id": str},
)

df_sensor = (
    df_sensor
    .sort_values(["machine_id", "timestamp"], ascending=True)
    .reset_index(drop=True)
)

# ----------------------------------------------------------
# [4] KONFIRMASI SETUP
# ----------------------------------------------------------
SEP = "=" * 65
sep = "-" * 65

print(SEP)
print("  SETUP KONFIRMASI")
print(SEP)
print(f"  SRC_PATH         : {SRC_PATH}")
print(f"  GLOBAL_SEED      : {GLOBAL_SEED}")
print(f"  SENSOR_FILE      : {SENSOR_FILE.name}  shape={df_sensor.shape}")
print(f"  W_CRITICAL_HRS   : {W_CRITICAL_HRS} jam")
print(f"  W_WARNING_HRS    : {W_WARNING_HRS} jam")
print(f"  LABEL_MAP        : {LABEL_MAP}")

# ============================================================
# IMPLEMENTASI TEMPORAL BACKWARD-LABELING
# ============================================================

def assign_health_labels(df, w_critical_hrs, w_warning_hrs):
    """
    Temporal backward-labeling per failure event.

    Window CRITICAL : [T_failure - w_critical_hrs, T_failure]
    Window WARNING  : [T_failure - w_warning_hrs,  T_failure - w_critical_hrs)
    WARNING tidak pernah menimpa CRITICAL.
    """
    df = df.copy()
    df["health_label"] = "HEALTHY"

    df_failures = df[df["failure"] == 1]

    for _, row in df_failures.iterrows():
        machine_id       = row["machine_id"]
        T_failure        = row["timestamp"]
        T_critical_start = T_failure - pd.Timedelta(hours=w_critical_hrs)
        T_warning_start  = T_failure - pd.Timedelta(hours=w_warning_hrs)

        # Masker CRITICAL
        mask_critical = (
            (df["machine_id"] == machine_id) &
            (df["timestamp"]  >= T_critical_start) &
            (df["timestamp"]  <= T_failure)
        )
        df.loc[mask_critical, "health_label"] = "CRITICAL"

        # Masker WARNING - JANGAN timpa CRITICAL
        mask_warning = (
            (df["machine_id"]   == machine_id) &
            (df["timestamp"]    >= T_warning_start) &
            (df["timestamp"]    <  T_critical_start) &
            (df["health_label"] != "CRITICAL")
        )
        df.loc[mask_warning, "health_label"] = "WARNING"

    return df


# ----------------------------------------------------------
# EKSEKUSI
# ----------------------------------------------------------
df_labeled = assign_health_labels(df_sensor, W_CRITICAL_HRS, W_WARNING_HRS)

# ============================================================
# LAPORAN VALIDASI
# ============================================================
print(f"\n{SEP}")
print("  LAPORAN VALIDASI LABEL ENGINEERING")
print(SEP)

# --- [A] Distribusi health_label ---
print(f"\n{sep}")
print("  [A] DISTRIBUSI health_label")
print(sep)

label_counts = df_labeled["health_label"].value_counts()
label_pct    = (df_labeled["health_label"].value_counts(normalize=True) * 100).round(4)

dist_df = pd.DataFrame({
    "label" : label_counts.index,
    "count" : label_counts.values,
    "pct_%" : label_pct.values,
})
print()
print(dist_df.to_string(index=False))

# --- [B] Konfirmasi: tidak ada CRITICAL yang tertimpa WARNING ---
print(f"\n{sep}")
print("  [B] KONFIRMASI INTEGRITAS LABEL")
print(sep)

n_failure_rows   = int((df_labeled["failure"] == 1).sum())
n_critical_rows  = int((df_labeled["health_label"] == "CRITICAL").sum())
n_fail_critical  = int(
    ((df_labeled["failure"] == 1) & (df_labeled["health_label"] == "CRITICAL")).sum()
)

flag = "[OK]" if n_failure_rows == n_fail_critical else "[WARN]"

print(f"\n  Total baris failure=1                      : {n_failure_rows}")
print(f"  Baris failure=1 berlabel CRITICAL          : {n_fail_critical}")
print(f"  Total baris berlabel CRITICAL (semua baris): {n_critical_rows}")
print(f"  {flag} Semua failure=1 berlabel CRITICAL   : {n_failure_rows == n_fail_critical}")

# --- [C] Ringkasan per machine_id ---
print(f"\n{sep}")
print("  [C] RINGKASAN PER MESIN")
print(sep)

per_machine = (
    df_labeled
    .groupby("machine_id")["health_label"]
    .value_counts()
    .unstack(fill_value=0)
    .rename_axis(None, axis=1)
    .reset_index()
)

for lbl in ["HEALTHY", "WARNING", "CRITICAL"]:
    if lbl not in per_machine.columns:
        per_machine[lbl] = 0

per_machine = (
    per_machine[["machine_id", "HEALTHY", "WARNING", "CRITICAL"]]
    .rename(columns={
        "HEALTHY" : "n_HEALTHY",
        "WARNING" : "n_WARNING",
        "CRITICAL": "n_CRITICAL",
    })
    .sort_values("machine_id", ascending=True)
)

print()
print(per_machine.to_string(index=False))

print(f"\n{SEP}")
print("  [OK] Label Engineering selesai. df_labeled siap untuk Fase 4.")
print(SEP)

In [ ]:
# ============================================================
# FASE 3 - SENSOR CONFIRMATION LAYER & EXPORT
# Cell 2: Validasi berbasis sensor + encode label + simpan parquet
# ============================================================

from config import DATA_INTERIM_DIR

SEP = "=" * 65
sep = "-" * 65

# ----------------------------------------------------------
# PARAMETER CONFIRMATION LAYER
# ----------------------------------------------------------
HIGH_PRIORITY_SENSORS    = [
    "temperature",
    "vibration",
    "pressure",
    "rpm",
    "power_consumption",
    "noise_level",
]
MIN_SENSORS_TRIGGERED    = 2
CONFIRMATION_PERCENTILE  = 90

# Hanya gunakan sensor yang benar-benar ada di df_labeled (defensive)
sensors_ok = [s for s in HIGH_PRIORITY_SENSORS if s in df_labeled.columns]

# ============================================================
# LANGKAH 1: Hitung threshold persentil dari baseline HEALTHY
# ============================================================
df_healthy_baseline = df_labeled[df_labeled["health_label"] == "HEALTHY"]

thresholds_p90 = {}
for sensor in sensors_ok:
    thresholds_p90[sensor] = df_healthy_baseline[sensor].quantile(
        CONFIRMATION_PERCENTILE / 100
    )

# ============================================================
# LANGKAH 2: Print tabel threshold
# ============================================================
print(SEP)
print("  SENSOR CONFIRMATION LAYER")
print(SEP)
print(f"  Percentile baseline    : P{CONFIRMATION_PERCENTILE} (dari distribusi HEALTHY)")
print(f"  Min sensors triggered  : {MIN_SENSORS_TRIGGERED}")
print(f"  Sensors digunakan      : {sensors_ok}")

print(f"\n{sep}")
print("  THRESHOLD P90 PER SENSOR (dari baseline HEALTHY)")
print(sep)
print()

thresh_df = pd.DataFrame(
    [{"sensor": k, "threshold_p90": round(v, 4)} for k, v in thresholds_p90.items()]
)
print(thresh_df.to_string(index=False))

# ============================================================
# LANGKAH 3 & 4: Fungsi confirm_label + apply
# ============================================================

def confirm_label(row):
    """
    Konfirmasi label WARNING/CRITICAL menggunakan sensor evidence.

    - HEALTHY    : tidak diubah
    - WARNING/CRITICAL : pertahankan jika >= MIN_SENSORS_TRIGGERED
                         sensor melewati threshold P90-nya,
                         downgrade ke HEALTHY jika kurang.
    """
    if row["health_label"] == "HEALTHY":
        return "HEALTHY"

    count = 0
    for sensor, threshold in thresholds_p90.items():
        if row[sensor] > threshold:
            count += 1

    if count >= MIN_SENSORS_TRIGGERED:
        return row["health_label"]
    else:
        return "HEALTHY"


df_labeled["health_label_confirmed"] = df_labeled.apply(confirm_label, axis=1)

# ============================================================
# LANGKAH 5: Laporan perbandingan sebelum vs sesudah
# ============================================================
print(f"\n{SEP}")
print("  LAPORAN PERBANDINGAN: SEBELUM vs SESUDAH KONFIRMASI")
print(SEP)

# Sebelum
before_counts = df_labeled["health_label"].value_counts()
before_pct    = (df_labeled["health_label"].value_counts(normalize=True) * 100).round(4)

# Sesudah
after_counts  = df_labeled["health_label_confirmed"].value_counts()
after_pct     = (df_labeled["health_label_confirmed"].value_counts(normalize=True) * 100).round(4)

print(f"\n{sep}")
print("  SEBELUM KONFIRMASI (health_label)")
print(sep)
print()
for lbl in ["HEALTHY", "WARNING", "CRITICAL"]:
    cnt = before_counts.get(lbl, 0)
    pct = before_pct.get(lbl, 0.0)
    print(f"    {lbl:<12} : {cnt:>7,}  ({pct:.4f}%)")

print(f"\n{sep}")
print("  SESUDAH KONFIRMASI (health_label_confirmed)")
print(sep)
print()
for lbl in ["HEALTHY", "WARNING", "CRITICAL"]:
    cnt = after_counts.get(lbl, 0)
    pct = after_pct.get(lbl, 0.0)
    print(f"    {lbl:<12} : {cnt:>7,}  ({pct:.4f}%)")

# Downgrade stats
n_downgraded = int(
    ((df_labeled["health_label"] != "HEALTHY") &
     (df_labeled["health_label_confirmed"] == "HEALTHY")).sum()
)
n_nonhealthy = int((df_labeled["health_label"] != "HEALTHY").sum())
pct_downgraded = (n_downgraded / n_nonhealthy * 100) if n_nonhealthy > 0 else 0.0

print(f"\n{sep}")
print("  STATISTIK DOWNGRADE")
print(sep)
print(f"\n  Baris WARNING/CRITICAL sebelum konfirmasi : {n_nonhealthy:,}")
print(f"  Di-downgrade ke HEALTHY                    : {n_downgraded:,}")
print(f"  Persentase downgrade                        : {pct_downgraded:.2f}%")

# ============================================================
# LANGKAH 6: Encoding label
# ============================================================
df_labeled["health_label_encoded"] = (
    df_labeled["health_label_confirmed"].map(LABEL_MAP)
)

print(f"\n{sep}")
print("  ENCODING health_label_confirmed -> health_label_encoded")
print(sep)
print(f"\n  LABEL_MAP : {LABEL_MAP}")
print(f"  NaN setelah encoding : {df_labeled['health_label_encoded'].isna().sum()}")

# ============================================================
# LANGKAH 7: Export ke parquet
# ============================================================
DATA_INTERIM_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_PATH = DATA_INTERIM_DIR / "df_sensor_labeled.parquet"

df_labeled.to_parquet(EXPORT_PATH, index=False)

# ============================================================
# LANGKAH 8: Konfirmasi export
# ============================================================
file_size_kb = EXPORT_PATH.stat().st_size / 1024

print(f"\n{SEP}")
print("  EXPORT KONFIRMASI")
print(SEP)
print(f"\n  Path disimpan  : {EXPORT_PATH}")
print(f"  Shape          : {df_labeled.shape}")
print(f"  Ukuran file    : {file_size_kb:.1f} KB")
print(f"  Kolom baru     : health_label | health_label_confirmed | health_label_encoded")
print(f"\n{SEP}")
print("  [OK] Sensor Confirmation Layer & Export selesai.")
print(SEP)

In [ ]:
# ============================================================
# FASE 3 - DATA PREVIEW & EXPORT CSV
# Cell 3: Preview per kelas + export CSV untuk inspeksi manual
# ============================================================

import pandas as pd
import sys
from pathlib import Path
from IPython.display import display

# ----------------------------------------------------------
# Pastikan SRC_PATH tersedia (cell dapat dijalankan mandiri)
# ----------------------------------------------------------
NOTEBOOK_DIR = Path().resolve()
ML_ROOT      = NOTEBOOK_DIR.parent.parent
SRC_PATH     = ML_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from config import DATA_INTERIM_DIR

SEP = "=" * 65
sep = "-" * 65

PARQUET_PATH = DATA_INTERIM_DIR / "df_sensor_labeled.parquet"

PREVIEW_COLS = [
    "timestamp",
    "machine_id",
    "temperature",
    "vibration",
    "noise_level",
    "failure",
    "health_label",
    "health_label_confirmed",
    "health_label_encoded",
]

PREVIEW_LABELS = ["HEALTHY", "WARNING", "CRITICAL"]

# ============================================================
# OPSI A - Preview per Kelas
# ============================================================

# [1] Baca ulang dari parquet
df_check = pd.read_parquet(PARQUET_PATH)

# [2] Info dasar
print(SEP)
print("  DATA PREVIEW - df_sensor_labeled.parquet")
print(SEP)
print(f"\n  Shape  : {df_check.shape}")
print("  Kolom  :")
for col in df_check.columns:
    print(f"    - {col}")

# Hanya gunakan kolom yang tersedia di file (defensive)
cols_ok = [c for c in PREVIEW_COLS if c in df_check.columns]

# [3] Sampel 3 baris per label
for label in PREVIEW_LABELS:
    subset = df_check[df_check["health_label_confirmed"] == label]
    print(f"\n{sep}")
    print(f"  SAMPEL KELAS: {label}  (total: {len(subset):,} baris)")
    print(sep)
    display(subset[cols_ok].head(3))

# ============================================================
# OPSI B - Export CSV Preview
# ============================================================

# [4] Simpan ke CSV
CSV_PATH = DATA_INTERIM_DIR / "df_sensor_labeled_preview.csv"
df_check.to_csv(CSV_PATH, index=False)

# [5] Konfirmasi export
file_size_mb = CSV_PATH.stat().st_size / (1024 * 1024)

print(f"\n{SEP}")
print("  EXPORT CSV KONFIRMASI")
print(SEP)
print(f"\n  Path CSV : {CSV_PATH}")
print(f"  Shape    : {df_check.shape}")
print(f"  Ukuran   : {file_size_mb:.2f} MB")

# [6] Catatan
print(f"\n{sep}")
print("  CATATAN")
print(sep)
print("\n  CATATAN: File CSV ini hanya untuk preview manual.")
print("  Pipeline resmi tetap menggunakan .parquet")
print(f"\n{SEP}")
print("  [OK] Data Preview & Export CSV selesai.")
print(SEP)